In [2]:
import os

os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('VECLIB_MAXIMUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

from pathlib import Path
import json
import random
import re
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display


sns.set_style('white')
plt.rcParams['figure.dpi'] = 120

PALETTE = {
    'neutral': '#4c7c78',
    'incorrect_suggestion': '#d4651a',
    'doubt_correct': '#73b3ab',
    'suggest_correct': '#0f6b6f',
}
SPLIT_ORDER = ['train', 'val', 'test']
TEMPLATE_ORDER = ['neutral', 'incorrect_suggestion', 'doubt_correct', 'suggest_correct']
RUN_DIR_INPUT = Path('/Users/itaishapira/Desktop/R1_knowledge_gap/LLMsKnow/results/sycophancy_bias_probe/mistralai_Mistral_7B_Instruct_v0_2/legacy_unknown/fast_dirty_mistral7b_instruct_V2')
ARTIFACTS = [
    'run_config.json',
    'sampling_manifest.json',
    'sampled_responses.csv',
    'final_tuples.csv',
    'summary_by_question.csv',
    'probe_metadata.json',
    'status.json',
    'sampling_records.jsonl',
]


def read_jsonl(path):
    rows = []
    with Path(path).open('r', encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def resolve_run_dir(path):
    if all((path / name).exists() for name in ARTIFACTS):
        return path
    nested = path / path.name
    if all((nested / name).exists() for name in ARTIFACTS):
        return nested
    raise FileNotFoundError(f'Could not find a complete run directory under {path}')


def resolve_bias_types(arg):
    return [item.strip() for item in str(arg).split(',') if item.strip()]


def try_load_tokenizer(model_name, cache_dir):
    try:
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            cache_dir=cache_dir,
            local_files_only=True,
            use_fast=True,
        )
        return tokenizer, 'exact model-token counts'
    except Exception:
        return None, 'whitespace-token approximation'


def token_len(text):
    if pd.isna(text):
        return np.nan
    text = str(text)
    if tokenizer is not None:
        return len(tokenizer(text, add_special_tokens=False).input_ids)
    return len(re.findall(r'\\S+', text))


def looks_truncated(text):
    if not isinstance(text, str):
        return False
    stripped = text.rstrip()
    if len(stripped) < 40:
        return False
    if re.search(r'[:,;\\-]\\s*$', stripped):
        return True
    if re.search(r'\\b(?:in|of|to|for|with|and|or|the|a|an|being|was|were)\\s*$', stripped.lower()):
        return True
    if re.search(r'\\b\\d{1,3}$', stripped):
        return True
    return stripped[-1] not in '.!?\"\'”’)]'


def pct(value):
    return f'{100 * float(value):.1f}%'


def show_qa(question, answer):
    display(Markdown(f'**Question.** {question}'))
    display(Markdown(f'**Answer.** {answer}'))


def finish_axis(ax, title, xlabel, ylabel, legend=False):
    ax.set_title(title, fontsize=18)
    ax.set_xlabel(xlabel, fontsize=15)
    ax.set_ylabel(ylabel, fontsize=15)
    ax.tick_params(axis='both', labelsize=12)
    if legend and ax.legend_ is not None:
        ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=2, frameon=False)


RUN_DIR = resolve_run_dir(RUN_DIR_INPUT)
REPO_ROOT = RUN_DIR.parents[3]
run_config = json.loads((RUN_DIR / 'run_config.json').read_text(encoding='utf-8'))
sampling_manifest = json.loads((RUN_DIR / 'sampling_manifest.json').read_text(encoding='utf-8'))
probe_metadata = json.loads((RUN_DIR / 'probe_metadata.json').read_text(encoding='utf-8'))
status = json.loads((RUN_DIR / 'status.json').read_text(encoding='utf-8'))
run_log = (RUN_DIR / 'run.log').read_text(encoding='utf-8') if (RUN_DIR / 'run.log').exists() else ''
sampled = pd.read_csv(RUN_DIR / 'sampled_responses.csv')
tuples = pd.read_csv(RUN_DIR / 'final_tuples.csv')
summary = pd.read_csv(RUN_DIR / 'summary_by_question.csv')
sampling_records = read_jsonl(RUN_DIR / 'sampling_records.jsonl')
bias_types = resolve_bias_types(run_config['bias_types'])
tokenizer, token_mode = try_load_tokenizer(run_config['model'], run_config.get('hf_cache_dir'))

sampled['response_token_len'] = sampled['response_raw'].map(token_len)
sampled['response_char_len'] = sampled['response_raw'].fillna('').str.len()
sampled['looks_truncated'] = sampled['response_raw'].map(looks_truncated)
sampled['pair_id'] = (
    sampled['split'].astype(str)
    + '||'
    + sampled['question_id'].astype(str)
    + '||'
    + sampled['draw_idx'].astype(str)
)
summary['delta_acc'] = summary['mean_C_xprime'] - summary['mean_C_x']
tuples['delta_correctness'] = tuples['C_xprime_yprime'] - tuples['C_x_y']

question_catalog = sampled[
    ['split', 'question_id', 'question', 'correct_answer', 'incorrect_answer']
].drop_duplicates()
prompt_catalog = sampled[
    [
        'split',
        'question_id',
        'template_type',
        'question',
        'correct_answer',
        'incorrect_answer',
        'prompt_template',
        'prompt_text',
    ]
].drop_duplicates()

raw_data_path = REPO_ROOT / run_config['data_dir'] / run_config['input_jsonl']
if not raw_data_path.exists():
    raw_data_path = Path(run_config['data_dir']) / run_config['input_jsonl']
raw_rows = read_jsonl(raw_data_path)
dedup_rows = deduplicate_rows(raw_rows)
all_groups = build_question_groups(dedup_rows, selected_bias_types=bias_types)
selected_groups = list(all_groups)
if run_config.get('max_questions') is not None:
    rng = random.Random(run_config['split_seed'])
    rng.shuffle(selected_groups)
    selected_groups = selected_groups[: int(run_config['max_questions'])]

rows_by_question = defaultdict(list)
for row in dedup_rows:
    rows_by_question[question_key(row)].append(row)

drop_reason_rows = []
for qkey, rows in rows_by_question.items():
    rows_by_type = {}
    for row in rows:
        row_type = template_type(row)
        if row_type is not None:
            rows_by_type[row_type] = row
    reasons = []
    if 'neutral' not in rows_by_type:
        reasons.append('missing_neutral')
    missing_bias = [bias for bias in bias_types if bias not in rows_by_type]
    if missing_bias:
        reasons.append('missing_bias_variant')
    neutral_row = rows_by_type.get('neutral')
    if neutral_row is not None:
        gold_answers = extract_gold_answers_from_base((neutral_row.get('base') or {}))
        if not gold_answers:
            reasons.append('missing_gold_answers')
        prompt_messages = neutral_row.get('prompt', [])
        if not isinstance(prompt_messages, list) or not prompt_messages:
            reasons.append('empty_neutral_prompt')
    for bias in bias_types:
        row = rows_by_type.get(bias)
        if row is not None:
            prompt_messages = row.get('prompt', [])
            if not isinstance(prompt_messages, list) or not prompt_messages:
                reasons.append(f'empty_prompt:{bias}')
    if not reasons:
        reasons = ['retained']
    for reason in reasons:
        drop_reason_rows.append({'reason': reason, 'question_key': qkey[0]})
drop_reason_df = pd.DataFrame(drop_reason_rows)

bias_rows = sampled[sampled['template_type'] != 'neutral'][
    ['split', 'question_id', 'draw_idx', 'template_type', 'usable_for_metrics', 'correctness']
].rename(
    columns={
        'template_type': 'bias_type',
        'usable_for_metrics': 'bias_usable',
        'correctness': 'bias_correctness',
    }
)
neutral_rows = sampled[sampled['template_type'] == 'neutral'][
    ['split', 'question_id', 'draw_idx', 'usable_for_metrics', 'correctness']
].rename(
    columns={
        'usable_for_metrics': 'neutral_usable',
        'correctness': 'neutral_correctness',
    }
)
pair_eval = bias_rows.merge(
    neutral_rows,
    on=['split', 'question_id', 'draw_idx'],
    how='left',
)
pair_eval['pair_usable'] = pair_eval['bias_usable'] & pair_eval['neutral_usable']
pair_eval['delta_correctness'] = pair_eval['bias_correctness'] - pair_eval['neutral_correctness']

sample_questions = question_catalog.sample(
    min(6, len(question_catalog)),
    random_state=0,
).sort_values(['split', 'question_id']).reset_index(drop=True)
show_qa(
    'Which questions should I sample to make sure the dataset looks sensible, diverse, and correctly paired with correct_answer and incorrect_answer?',
    f'This notebook resolved the run directory to {RUN_DIR}. The smoke-test run contains {len(question_catalog)} unique question groups sampled from {len(selected_groups)} selected groups. The table below shows six question/answer pairs across splits for a quick data sanity pass.',
)
display(sample_questions[['split', 'question_id', 'question', 'correct_answer', 'incorrect_answer']])


FileNotFoundError: [Errno 2] No such file or directory: 'data/sycophancy-eval/answer.jsonl'

In [ ]:
assert 'prompt_catalog' in globals(), 'Run cell 1 first.'

prompt_question_id = prompt_catalog[['question_id']].drop_duplicates().sample(1, random_state=1).iloc[0, 0]
prompt_rows = prompt_catalog[prompt_catalog['question_id'] == prompt_question_id].sort_values('template_type').copy()
prompt_rows['added_bias_clause'] = prompt_rows.apply(
    lambda row: str(row['prompt_text']).replace(str(row['question']), '', 1).strip(),
    axis=1,
)
show_qa(
    'Which prompts should I sample for the same question to confirm that the neutral and biased versions differ only by the intended bias text?',
    f'Question {prompt_question_id} has {len(prompt_rows)} prompt variants in this run. The neutral prompt is just the base question, and the three biased prompts add the expected bias clauses shown below.',
)
display(prompt_rows[['template_type', 'prompt_template', 'added_bias_clause', 'prompt_text']].reset_index(drop=True))


In [ ]:
assert 'sampled' in globals(), 'Run cell 1 first.'

variant = prompt_catalog[['split', 'question_id', 'template_type']].drop_duplicates().sample(1, random_state=2).iloc[0]
variant_split = variant['split']
variant_qid = variant['question_id']
variant_template = variant['template_type']
response_view = sampled[
    (sampled['split'] == variant_split)
    & (sampled['question_id'] == variant_qid)
    & (sampled['template_type'] == variant_template)
].sort_values('draw_idx').copy()
mean_len = response_view['response_token_len'].mean()
trunc_count = int(response_view['looks_truncated'].sum())
show_qa(
    'For one randomly chosen prompt, what do several raw model responses look like, how were they parsed/labeled, and how long are they?',
    f'Selected {variant_template} for {variant_qid} in the {variant_split} split. Across {len(response_view)} draws, mean completion length was {mean_len:.1f} using {token_mode}; {trunc_count} draw(s) triggered the truncation heuristic.',
)
display(
    response_view[
        [
            'draw_idx',
            'prompt_text',
            'response_raw',
            'response',
            'correctness',
            'grading_status',
            'grading_reason',
            'response_token_len',
            'looks_truncated',
        ]
    ].reset_index(drop=True)
)


In [ ]:
assert 'sampled' in globals(), 'Run cell 1 first.'

max_new_tokens = int(run_config['max_new_tokens'])
exact_counts = tokenizer is not None
if exact_counts:
    near_limit_mask = sampled['response_token_len'] >= (max_new_tokens - 4)
    near_limit_text = f'{int(near_limit_mask.sum())} responses sit within four tokens of max_new_tokens={max_new_tokens}.'
else:
    near_limit_mask = sampled['looks_truncated']
    near_limit_text = 'Exact model-token counts were not available locally, so this cell falls back to a whitespace-token proxy plus a truncation heuristic.'
trunc_n = int(sampled['looks_truncated'].sum())
show_qa(
    'Are many responses hitting the length ceiling, and do any look cut off in the middle because of the small max_new_tokens?',
    f'{near_limit_text} Separately, {trunc_n} of {len(sampled)} responses ({pct(trunc_n / len(sampled))}) look truncated by the text heuristic. The histogram below uses {token_mode}.',
)
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.histplot(
    data=sampled,
    x='response_token_len',
    hue='template_type',
    hue_order=TEMPLATE_ORDER,
    bins=18,
    multiple='stack',
    palette=[PALETTE[key] for key in TEMPLATE_ORDER],
    ax=ax,
)
ax.axvline(max_new_tokens, color='black', linestyle='--', linewidth=1.5, label='max_new_tokens')
finish_axis(ax, f'Completion length distribution ({token_mode})', 'Completion length', 'Responses', legend=True)
plt.show()
display(
    sampled.loc[
        sampled['looks_truncated'],
        ['split', 'question_id', 'template_type', 'draw_idx', 'response_raw', 'response_token_len'],
    ].head(8).reset_index(drop=True)
)


In [ ]:
assert 'sampled' in globals(), 'Run cell 1 first.'

usable = sampled[sampled['usable_for_metrics']].copy()
audit_parts = []
for template in TEMPLATE_ORDER:
    subset = usable[usable['template_type'] == template]
    if len(subset) == 0:
        continue
    audit_parts.append(subset.sample(min(2, len(subset)), random_state=5))
manual_audit = pd.concat(audit_parts, ignore_index=True).drop_duplicates('record_id').head(8).copy()
manual_audit['parsed_appears_in_raw'] = manual_audit.apply(
    lambda row: str(row['response']) in str(row['response_raw']),
    axis=1,
)
show_qa(
    'For a manual sample of labeled responses, do the parsed short answers actually match the raw generations and the assigned labels?',
    f'The table below is a manual-audit batch of {len(manual_audit)} usable responses spanning prompt types. It is meant for direct inspection: you can compare the raw generation, the parsed short answer, and the label side by side.',
)
display(
    manual_audit[
        ['split', 'template_type', 'question', 'response_raw', 'response', 'correctness', 'parsed_appears_in_raw']
    ].reset_index(drop=True)
)


In [ ]:
assert 'sampled' in globals(), 'Run cell 1 first.'

ambiguous = sampled[~sampled['usable_for_metrics']].copy()
reason_counts = ambiguous['grading_reason'].value_counts().rename_axis('grading_reason').reset_index(name='n')
template_rates = (
    sampled.groupby('template_type')['usable_for_metrics']
    .apply(lambda values: 1 - values.mean())
    .reindex(TEMPLATE_ORDER)
    .fillna(0)
    .reset_index(name='ambiguity_rate')
)
top_reason = reason_counts.iloc[0]['grading_reason'] if len(reason_counts) else 'none'
show_qa(
    'How many responses are ambiguous or unlabeled, what are the main ambiguity reasons, and do those examples look truly ambiguous?',
    f'{len(ambiguous)} of {len(sampled)} responses ({pct(len(ambiguous) / len(sampled))}) are unusable for metrics. The dominant reason is {top_reason}; examples are shown below for spot checking.',
)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(
    data=template_rates,
    x='template_type',
    y='ambiguity_rate',
    order=TEMPLATE_ORDER,
    palette=[PALETTE[key] for key in TEMPLATE_ORDER],
    ax=ax,
)
finish_axis(ax, 'Ambiguity rate by prompt type', 'Prompt type', 'Ambiguity rate')
plt.xticks(rotation=20, ha='right')
plt.show()
display(reason_counts)
display(
    ambiguous[
        ['split', 'question_id', 'template_type', 'draw_idx', 'response_raw', 'grading_reason']
    ].head(10).reset_index(drop=True)
)


In [ ]:
assert 'summary' in globals(), 'Run cell 1 first.'

accuracy_by_split_bias = summary.groupby(['split', 'bias_type'])[['mean_C_x', 'mean_C_xprime']].mean().reset_index()
accuracy_by_split_bias['delta'] = accuracy_by_split_bias['mean_C_xprime'] - accuracy_by_split_bias['mean_C_x']
accuracy_long = pd.concat(
    [
        accuracy_by_split_bias[['split', 'bias_type', 'mean_C_x']]
        .rename(columns={'mean_C_x': 'accuracy'})
        .assign(prompt_condition='neutral'),
        accuracy_by_split_bias[['split', 'bias_type', 'mean_C_xprime']]
        .rename(columns={'mean_C_xprime': 'accuracy'})
        .assign(prompt_condition='biased'),
    ],
    ignore_index=True,
)
overall_delta = accuracy_by_split_bias.groupby('bias_type')['delta'].mean().sort_values()
worst_bias = overall_delta.index[0]
show_qa(
    'What is the model\'s accuracy on neutral prompts versus biased prompts, for each bias type and per question?',
    f'Neutral accuracy is the baseline, and biased accuracy is usually lower in this smoke run. The largest mean drop across splits comes from {worst_bias} ({overall_delta.iloc[0]:.3f}). The table below keeps the split-level view.',
)
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(
    data=accuracy_long,
    x='bias_type',
    y='accuracy',
    hue='prompt_condition',
    order=bias_types,
    palette={'neutral': PALETTE['neutral'], 'biased': PALETTE['incorrect_suggestion']},
    ax=ax,
)
finish_axis(ax, 'Mean accuracy before and after bias', 'Bias type', 'Accuracy', legend=True)
plt.xticks(rotation=15, ha='right')
plt.show()
display(accuracy_by_split_bias.sort_values(['split', 'bias_type']).reset_index(drop=True))


In [ ]:
assert 'probe_metadata' in globals(), 'Run cell 1 first.'

probe_rows = []
for probe_name, meta in probe_metadata.items():
    template = 'neutral' if probe_name == 'probe_no_bias' else probe_name.replace('probe_bias_', '')
    train_subset = sampled[
        (sampled['split'] == 'train')
        & (sampled['template_type'] == template)
        & (sampled['usable_for_metrics'])
    ]
    val_subset = sampled[
        (sampled['split'] == 'val')
        & (sampled['template_type'] == template)
        & (sampled['usable_for_metrics'])
    ]
    probe_rows.append(
        {
            'probe_name': probe_name,
            'template_type': template,
            'usable_train_records': len(train_subset),
            'usable_val_records': len(val_subset),
            'best_layer': meta.get('best_layer'),
            'best_dev_auc': meta.get('best_dev_auc'),
            'trained_successfully': bool(meta.get('saved_best_model')),
        }
    )
probe_df = pd.DataFrame(probe_rows).sort_values('probe_name').reset_index(drop=True)
trained_n = int(probe_df['trained_successfully'].sum())
show_qa(
    'How many probes were trained, on how many usable examples, and what was the performance of each one?',
    f'This run wrote {len(probe_df)} probe entries. {trained_n} produced a retrained best model, while the others stopped early because of data availability or class-balance issues. The bar plot shows best validation AUC when available.',
)
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(
    data=probe_df,
    x='probe_name',
    y='best_dev_auc',
    color=PALETTE['doubt_correct'],
    ax=ax,
)
finish_axis(ax, 'Probe validation AUC by probe', 'Probe', 'Best dev AUC')
plt.xticks(rotation=20, ha='right')
plt.show()
display(probe_df)


In [ ]:
assert 'probe_df' in globals(), 'Run cell 8 first.'

class_rows = []
for template in bias_types:
    for split in ['train', 'val']:
        subset = sampled[
            (sampled['template_type'] == template)
            & (sampled['split'] == split)
            & (sampled['usable_for_metrics'])
        ]
        counts = subset['correctness'].value_counts().to_dict()
        class_rows.append(
            {
                'template_type': template,
                'split': split,
                'n_usable': len(subset),
                'n_correct': int(counts.get(1.0, 0) + counts.get(1, 0)),
                'n_incorrect': int(counts.get(0.0, 0) + counts.get(0, 0)),
            }
        )
class_df = pd.DataFrame(class_rows)
strategy_df = probe_df[probe_df['template_type'] != 'neutral'].copy()
show_qa(
    'Do probes trained for different bias strategies differ in performance when compared on the same evaluation subset?',
    'Nominally, the strategy probes differ, but this smoke run is not an apples-to-apples comparison. incorrect_suggestion has an all-incorrect validation set, and suggest_correct has an all-incorrect training set, so the missing AUCs are driven by class balance as much as by strategy.',
)
display(strategy_df[['template_type', 'usable_train_records', 'usable_val_records', 'best_layer', 'best_dev_auc', 'trained_successfully']])
display(class_df)


In [ ]:
assert 'probe_metadata' in globals(), 'Run cell 1 first.'

layer_rows = []
for probe_name, meta in probe_metadata.items():
    for layer, auc in (meta.get('auc_per_layer') or {}).items():
        layer_rows.append(
            {
                'probe_name': probe_name,
                'layer': int(layer),
                'auc': pd.to_numeric(auc, errors='coerce'),
            }
        )
layer_auc_df = pd.DataFrame(layer_rows).sort_values(['probe_name', 'layer']).reset_index(drop=True)
selected_layers = {
    probe_name: meta.get('best_layer')
    for probe_name, meta in probe_metadata.items()
    if meta.get('best_layer') is not None
}
boundary_hits = [
    probe_name
    for probe_name, layer in selected_layers.items()
    if int(layer) in {int(run_config['probe_layer_min']), int(run_config['probe_layer_max'])}
]
probe_palette = {
    'probe_no_bias': PALETTE['neutral'],
    'probe_bias_incorrect_suggestion': PALETTE['incorrect_suggestion'],
    'probe_bias_doubt_correct': PALETTE['doubt_correct'],
    'probe_bias_suggest_correct': PALETTE['suggest_correct'],
}
show_qa(
    'Do probe results differ by layer enough to change which layers are worth analyzing in later rounds?',
    f'Yes. The successful probes select different layers: {selected_layers}. Boundary hits are {boundary_hits if boundary_hits else "none"}, which is a useful sanity check for whether the searched layer range is wide enough.',
)
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.lineplot(
    data=layer_auc_df,
    x='layer',
    y='auc',
    hue='probe_name',
    marker='o',
    palette=probe_palette,
    ax=ax,
)
finish_axis(ax, 'Validation AUC by layer', 'Layer', 'Validation AUC', legend=True)
plt.show()
display(layer_auc_df.pivot(index='layer', columns='probe_name', values='auc'))


In [ ]:
assert 'raw_rows' in globals(), 'Run cell 1 first.'

dataset_counts = pd.DataFrame(
    {
        'stage': ['raw_rows', 'deduplicated_rows', 'complete_question_groups', 'smoke_test_selected_groups'],
        'count': [len(raw_rows), len(dedup_rows), len(all_groups), len(selected_groups)],
    }
)
show_qa(
    'How many raw rows exist per prompt type, how many survive deduplication, and how many complete question groups remain after requiring all prompt variants?',
    f'The source file has {len(raw_rows)} rows. Dedup removed {len(raw_rows) - len(dedup_rows)} rows, leaving {len(all_groups)} complete question groups; the smoke-test run sampled {len(selected_groups)} of them.',
)
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(
    data=dataset_counts,
    x='stage',
    y='count',
    palette=[PALETTE['neutral'], PALETTE['doubt_correct'], PALETTE['suggest_correct'], PALETTE['incorrect_suggestion']],
    ax=ax,
)
finish_axis(ax, 'Dataset size through the filtering pipeline', 'Stage', 'Count')
plt.xticks(rotation=20, ha='right')
plt.show()
display(dataset_counts)


In [ ]:
assert 'drop_reason_df' in globals(), 'Run cell 1 first.'

reason_order = [
    'missing_neutral',
    'missing_bias_variant',
    'missing_gold_answers',
    'empty_neutral_prompt',
    'empty_prompt:incorrect_suggestion',
    'empty_prompt:doubt_correct',
    'empty_prompt:suggest_correct',
    'retained',
]
drop_reason_summary = pd.DataFrame(
    {
        'reason': reason_order,
        'n_questions': [int((drop_reason_df['reason'] == reason).sum()) for reason in reason_order],
    }
)
dropped_total = int(drop_reason_summary.loc[drop_reason_summary['reason'] != 'retained', 'n_questions'].sum())
retained_total = int(drop_reason_summary.loc[drop_reason_summary['reason'] == 'retained', 'n_questions'].sum())
show_qa(
    'How many questions are dropped because a prompt variant is missing, because gold answers cannot be extracted, or because the question-answer pair is malformed?',
    f'For the failure modes checked here, this dataset shows {dropped_total} dropped question groups before the smoke-test cap. All {retained_total} recognized groups were retained; the later reduction to 24 questions comes from the smoke-test limit, not from malformed data.',
)
display(drop_reason_summary)


In [ ]:
assert 'raw_rows' in globals(), 'Run cell 1 first.'

duplicate_rows_removed = len(raw_rows) - len(dedup_rows)
dup_rows = pd.DataFrame(
    [
        {
            'question': question_key(row)[0],
            'correct_answer': question_key(row)[1],
            'incorrect_answer': question_key(row)[2],
            'template_type': template_type(row),
        }
        for row in raw_rows
    ]
)
dup_summary = (
    dup_rows.groupby(['question', 'correct_answer', 'incorrect_answer', 'template_type'])
    .size()
    .reset_index(name='raw_count')
)
dup_summary = dup_summary[dup_summary['raw_count'] > 1].sort_values('raw_count', ascending=False)
show_qa(
    'Are there duplicate questions or duplicate question-answer pairs that might overweight particular items?',
    f'Dedup removed {duplicate_rows_removed} rows from the raw file. The table below shows the question/template combinations that appeared more than once before deduplication.',
)
display(dup_summary.head(10).reset_index(drop=True))


In [ ]:
assert 'sampling_manifest' in globals(), 'Run cell 1 first.'

train_ids = set(sampling_manifest['sampling_spec']['train_question_ids'])
val_ids = set(sampling_manifest['sampling_spec']['val_question_ids'])
test_ids = set(sampling_manifest['sampling_spec']['test_question_ids'])
split_question_counts = question_catalog.groupby('split')['question_id'].nunique().reindex(SPLIT_ORDER).reset_index(name='n_questions')
split_template_counts = sampled.groupby(['split', 'template_type']).size().reset_index(name='n_responses')
show_qa(
    'Are train, val, and test truly disjoint at the question level, and are bias types reasonably balanced across those splits?',
    f'Question-ID overlap is train/val={len(train_ids & val_ids)}, train/test={len(train_ids & test_ids)}, val/test={len(val_ids & test_ids)}. The response-count bars below should line up with 4 prompt variants x 2 draws per selected question inside each split.',
)
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(
    data=split_template_counts,
    x='split',
    y='n_responses',
    hue='template_type',
    order=SPLIT_ORDER,
    hue_order=TEMPLATE_ORDER,
    palette=[PALETTE[key] for key in TEMPLATE_ORDER],
    ax=ax,
)
finish_axis(ax, 'Response counts by split and prompt type', 'Split', 'Responses', legend=True)
plt.show()
display(split_question_counts)
display(split_template_counts.sort_values(['split', 'template_type']).reset_index(drop=True))


In [ ]:
assert 'prompt_catalog' in globals(), 'Run cell 1 first.'

consistency = sampled.groupby(['split', 'question_id'])[['correct_answer', 'incorrect_answer']].nunique().reset_index()
inconsistent_questions = consistency[
    (consistency['correct_answer'] > 1) | (consistency['incorrect_answer'] > 1)
].copy()
prompt_checks = prompt_catalog.copy()
prompt_checks['contains_correct'] = prompt_checks.apply(
    lambda row: str(row['correct_answer']) in str(row['prompt_text']),
    axis=1,
)
prompt_checks['contains_incorrect'] = prompt_checks.apply(
    lambda row: str(row['incorrect_answer']) in str(row['prompt_text']),
    axis=1,
)
prompt_checks['prompt_ok'] = False
neutral_mask = prompt_checks['template_type'] == 'neutral'
incorrect_mask = prompt_checks['template_type'] == 'incorrect_suggestion'
doubt_mask = prompt_checks['template_type'] == 'doubt_correct'
suggest_mask = prompt_checks['template_type'] == 'suggest_correct'
prompt_checks.loc[neutral_mask, 'prompt_ok'] = (
    prompt_checks.loc[neutral_mask, 'prompt_text'].str.strip()
    == prompt_checks.loc[neutral_mask, 'question'].str.strip()
)
prompt_checks.loc[incorrect_mask, 'prompt_ok'] = (
    prompt_checks.loc[incorrect_mask, 'contains_incorrect']
    & ~prompt_checks.loc[incorrect_mask, 'contains_correct']
)
prompt_checks.loc[doubt_mask, 'prompt_ok'] = prompt_checks.loc[doubt_mask, 'contains_correct']
prompt_checks.loc[suggest_mask, 'prompt_ok'] = prompt_checks.loc[suggest_mask, 'contains_correct']
bad_prompts = prompt_checks[~prompt_checks['prompt_ok']].copy()
show_qa(
    'For each question, are the correct_answer and incorrect_answer consistent across all prompt variants, with no accidental swaps?',
    f'{len(inconsistent_questions)} question groups change their answer pair across prompt variants, and {len(bad_prompts)} prompt rows fail the expected interpolation check.',
)
display(prompt_checks.groupby(['template_type', 'prompt_ok']).size().unstack(fill_value=0))
display(bad_prompts[['split', 'question_id', 'template_type', 'prompt_text', 'correct_answer', 'incorrect_answer']].head(10).reset_index(drop=True))


In [ ]:
assert 'sampled' in globals(), 'Run cell 1 first.'

ambiguous = sampled[~sampled['usable_for_metrics']].copy()
ambiguity_by_template = (
    sampled.groupby('template_type')['usable_for_metrics']
    .apply(lambda values: 1 - values.mean())
    .reindex(TEMPLATE_ORDER)
    .fillna(0)
    .reset_index(name='ambiguity_rate')
)
top_questions = (
    ambiguous.groupby(['split', 'question_id', 'question'])
    .size()
    .reset_index(name='n_ambiguous')
    .sort_values('n_ambiguous', ascending=False)
    .head(10)
)
worst_template = ambiguity_by_template.sort_values('ambiguity_rate', ascending=False).iloc[0]
show_qa(
    'Are ambiguous or no-label cases concentrated in certain answer formats, question types, or bias templates?',
    f'Yes, they are concentrated by prompt type. The highest ambiguity rate is for {worst_template["template_type"]} at {pct(worst_template["ambiguity_rate"])}. The table below shows the questions with the most ambiguous draws.',
)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(
    data=ambiguity_by_template,
    x='template_type',
    y='ambiguity_rate',
    order=TEMPLATE_ORDER,
    palette=[PALETTE[key] for key in TEMPLATE_ORDER],
    ax=ax,
)
finish_axis(ax, 'Ambiguity concentration by prompt type', 'Prompt type', 'Ambiguity rate')
plt.xticks(rotation=20, ha='right')
plt.show()
display(top_questions.reset_index(drop=True))


In [ ]:
assert 'pair_eval' in globals(), 'Run cell 1 first.'

pair_loss = pair_eval.copy()
pair_loss['loss_reason'] = np.select(
    [
        pair_loss['pair_usable'],
        (~pair_loss['neutral_usable']) & pair_loss['bias_usable'],
        pair_loss['neutral_usable'] & (~pair_loss['bias_usable']),
    ],
    ['usable', 'neutral_only_unusable', 'bias_only_unusable'],
    default='both_unusable',
)
pair_counts = pair_loss.groupby('bias_type')['pair_usable'].agg(total_pairs='size', usable_pairs='sum').reset_index()
pair_counts['lost_pairs'] = pair_counts['total_pairs'] - pair_counts['usable_pairs']
pair_counts_long = pd.concat(
    [
        pair_counts[['bias_type', 'usable_pairs']].rename(columns={'usable_pairs': 'n_pairs'}).assign(pair_status='usable'),
        pair_counts[['bias_type', 'lost_pairs']].rename(columns={'lost_pairs': 'n_pairs'}).assign(pair_status='lost'),
    ],
    ignore_index=True,
)
loss_summary = pair_loss.groupby(['bias_type', 'loss_reason']).size().reset_index(name='n_pairs')
show_qa(
    'When one side of a neutral/bias pair is ambiguous and the other is usable, how much data do we lose from pairwise analyses, and could that distort the estimated bias effect?',
    f'Out of {len(pair_loss)} possible neutral/bias draw pairs, {int(pair_loss["pair_usable"].sum())} are usable and {int((~pair_loss["pair_usable"]).sum())} are lost. The breakdown below shows whether the missing side is usually neutral, biased, or both.',
)
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(
    data=pair_counts_long,
    x='bias_type',
    y='n_pairs',
    hue='pair_status',
    order=bias_types,
    palette={'usable': PALETTE['doubt_correct'], 'lost': PALETTE['incorrect_suggestion']},
    ax=ax,
)
finish_axis(ax, 'Usable versus lost neutral/bias pairs', 'Bias type', 'Pairs', legend=True)
plt.xticks(rotation=15, ha='right')
plt.show()
display(loss_summary.sort_values(['bias_type', 'loss_reason']).reset_index(drop=True))


In [ ]:
assert 'summary' in globals(), 'Run cell 1 first.'

effect_df = summary.copy()
effect_df['delta_acc'] = effect_df['mean_C_xprime'] - effect_df['mean_C_x']
effect_df['neutral_bucket'] = pd.cut(
    effect_df['mean_C_x'],
    bins=[-0.01, 0.0, 0.25, 0.5, 0.75, 1.0],
    labels=['0.00', '0.25', '0.50', '0.75', '1.00'],
)
bucket_df = (
    effect_df.groupby(['neutral_bucket', 'bias_type'], observed=False)['delta_acc']
    .mean()
    .reset_index()
)
worst_bucket_rows = bucket_df.sort_values('delta_acc').dropna().head(3)
show_qa(
    'Are the largest bias effects coming from already-hard questions, or do they also appear on questions the model usually gets right?',
    'The strongest negative bias effects in this run happen on questions that were easy under the neutral prompt. The bar chart bins questions by neutral accuracy before measuring the mean change under bias.',
)
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.barplot(
    data=bucket_df,
    x='neutral_bucket',
    y='delta_acc',
    hue='bias_type',
    hue_order=bias_types,
    palette=[PALETTE[key] for key in bias_types],
    ax=ax,
)
ax.axhline(0, color='black', linewidth=1)
finish_axis(ax, 'Bias effect versus neutral-question difficulty', 'Neutral accuracy bucket', 'Mean biased - neutral accuracy', legend=True)
plt.show()
display(effect_df.sort_values('delta_acc').head(10)[['split', 'question_id', 'bias_type', 'mean_C_x', 'mean_C_xprime', 'delta_acc']].reset_index(drop=True))


In [ ]:
assert 'tuples' in globals(), 'Run cell 1 first.'

stability_df = (
    tuples.groupby(['split', 'question_id', 'bias_type'])['delta_correctness']
    .agg(draws='size', distinct_deltas='nunique', min_delta='min', max_delta='max', mean_delta='mean')
    .reset_index()
)
stability_df['is_stable'] = stability_df['distinct_deltas'] == 1
stable_rate = stability_df['is_stable'].mean()
stability_counts = stability_df.groupby(['bias_type', 'is_stable']).size().reset_index(name='n_groups')
unstable = stability_df[~stability_df['is_stable']].sort_values(['split', 'question_id', 'bias_type']).reset_index(drop=True)
show_qa(
    'Is the observed bias effect stable across multiple draws, or is it driven by a small number of noisy samples?',
    f'{int(stability_df["is_stable"].sum())} of {len(stability_df)} question/bias groups ({pct(stable_rate)}) have the same draw-level bias effect across both draws; {len(unstable)} groups are mixed across draws.',
)
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(
    data=stability_counts,
    x='bias_type',
    y='n_groups',
    hue='is_stable',
    order=bias_types,
    palette={True: PALETTE['doubt_correct'], False: PALETTE['incorrect_suggestion']},
    ax=ax,
)
finish_axis(ax, 'Draw-to-draw stability of the bias effect', 'Bias type', 'Question groups', legend=True)
plt.xticks(rotation=15, ha='right')
plt.show()
display(unstable)


In [ ]:
assert 'sampled' in globals(), 'Run cell 1 first.'

val_usable = sampled[(sampled['split'] == 'val') & (sampled['usable_for_metrics'])].copy()
val_usable_sets = {
    template: set(zip(df['question_id'], df['draw_idx']))
    for template, df in val_usable.groupby('template_type')
}
overlap_rows = []
for row_template in TEMPLATE_ORDER:
    for col_template in TEMPLATE_ORDER:
        row_set = val_usable_sets.get(row_template, set())
        col_set = val_usable_sets.get(col_template, set())
        union = len(row_set | col_set)
        intersection = len(row_set & col_set)
        overlap_rows.append(
            {
                'row_template': row_template,
                'col_template': col_template,
                'intersection': intersection,
                'jaccard': 0.0 if union == 0 else intersection / union,
            }
        )
overlap_df = pd.DataFrame(overlap_rows)
overlap_matrix = overlap_df.pivot(index='row_template', columns='col_template', values='jaccard')
off_diagonal = overlap_df[overlap_df['row_template'] != overlap_df['col_template']]['jaccard']
show_qa(
    'When comparing probes, strategies, or layers, am I always comparing them on the same usable set of examples rather than different filtered subsets?',
    f'No. Across usable validation examples, off-diagonal Jaccard overlap ranges from {off_diagonal.min():.2f} to {off_diagonal.max():.2f}, so probe comparisons are happening on similar but not identical subsets. This matters when reading cross-probe AUC differences in a very small run.',
)
fig, ax = plt.subplots(figsize=(6.5, 5.0))
sns.heatmap(
    overlap_matrix.loc[TEMPLATE_ORDER, TEMPLATE_ORDER],
    annot=True,
    fmt='.2f',
    cmap=sns.light_palette(PALETTE['doubt_correct'], as_cmap=True),
    cbar_kws={'label': 'Jaccard overlap'},
    ax=ax,
)
ax.set_title('Usable validation-subset overlap across prompt types', fontsize=18)
ax.set_xlabel('Validation subset B', fontsize=15)
ax.set_ylabel('Validation subset A', fontsize=15)
ax.tick_params(axis='both', labelsize=12)
plt.show()
display(overlap_df[['row_template', 'col_template', 'intersection', 'jaccard']])
